# Performance evaluation

In [ ]:
# general
from pathlib import Path
import pandas as pd

# plotting
import plotly.graph_objects as go
from   plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler # for scaling

from src.utils import load_config
cfg = load_config()

## Data

In [ ]:
new_names = {"Pearson-r": "r", "Beta-KGE": "Beta", "Alpha-NSE": "Gamma"}

# metrics
metrics = pd.read_csv("data/hydro/performance_metrics.csv", index_col=0)
metrics = metrics.rename(columns=new_names)
metrics["Beta"] = (metrics["Beta"]*100)-100

# metadata 
andeanGC_metadata = pd.read_csv(Path(cfg["dir"]["andeangc"]) / "dataset/AndeanGC_metadata.csv", index_col=0)
andeanGC_metadata = andeanGC_metadata.join(metrics.rename(columns=new_names), how="left").dropna(axis=0)
andeanGC_metadata['split'] = andeanGC_metadata['split'].str.capitalize()

## Plot (MS)

In [ ]:
fig = make_subplots(rows=2, cols=3, vertical_spacing=0.02, shared_xaxes=False, shared_yaxes=True,
                    horizontal_spacing=0.02, row_heights=[0.15, 0.9])

colorscale = [(0, '#fde725'), (0.1, '#6ece58'), (0.2, '#35b779'), (0.6, '#31688e'), (1, '#440154')]
color_line = {"Train": "#8D8D8D", "Validation": "#000000", "Test": "#000000"}
scaler = MinMaxScaler(feature_range=(8, 30))
opacity = 0.8

# Define metrics configuration
metrics = [
    {"col": 1, "name": "KGE (-)", "key": "KGE", "dtick": 0.3, "range": (-0.1, 1.05)},
    {"col": 2, "name": "r-Pearson (-)", "key": "r", "dtick": 0.3, "range": (-0.1, 1.05)},
    {"col": 3, "name": "Bias (%)", "key": "Beta", "dtick": 50, "range": (-100, 100)}
]

for sce in ["Train", "Validation", "Test"]:
    for metric in metrics:
        col = metric["col"]
        key = metric["key"]
        
        # Box plot (row 1)
        fig.add_trace(go.Box(x=andeanGC_metadata[andeanGC_metadata.split == sce][key], 
                             marker_color="#35b779", name=sce, 
                             offsetgroup=sce, showlegend=False), 
                      row=1, col=col)
        
        # Scatter plot (row 2)
        fig.add_trace(go.Scatter(
            x=andeanGC_metadata[andeanGC_metadata.split == sce][key], 
            y=andeanGC_metadata[andeanGC_metadata.split == sce].gauge_lat, 
            mode='markers', name=sce, showlegend=False,
            marker=dict(
                size=scaler.fit_transform(andeanGC_metadata[andeanGC_metadata.split == sce][['basin_area']]),
                color=andeanGC_metadata[andeanGC_metadata.split == sce].glacier_area_RGI60,
                colorscale=colorscale, line_width=1, line_color=color_line[sce],
                cmin=0, cmax=25, opacity=opacity, 
                showscale=(sce == 'Train' and col == 1),
                colorbar=dict(title="Glacier<br>area", x=1,thickness=18, tickwidth=1, ticklen =5, ticksuffix="%")
                    if (sce == 'Train' and col == 1) else None
            )), row=2, col=col)

# Add reference lines and update axes
for col in range(1, 4):
    metric = metrics[col - 1]
    for row in range(1, 3):
        fig.add_vline(x=1 if metric["key"] in ["KGE", "r"] else 0,
                      line_width=1, line_dash="dot", line_color="black", 
                      opacity=0.7, row=row, col=col)
    
    fig.update_xaxes(range=metric["range"], dtick=metric["dtick"], 
                     showticklabels=False, row=1, col=col)
    fig.update_xaxes(range=metric["range"], dtick=metric["dtick"], 
                     title=metric["name"], row=2, col=col)

fig.update_xaxes(ticks="outside", griddash="dot", zeroline=False)
fig.update_yaxes(ticks="outside", griddash="dot", zeroline=False)
fig.update_yaxes(title_standoff=0, ticksuffix="ºS",
                 range=(-56, -7), row=2, col=1)


fig.update_layout(
    annotations=[dict(x=0, y=0.84, xref="paper", yref="paper", 
                     text="<span style='color:black'>○</span> Validation / Test<br><span style='color:grey'>○</span> Train",
                     showarrow=False, font=dict(size=15), bgcolor="rgba(255,255,255,0.8)", 
                     borderwidth=0, align="left")],
    legend=dict(bgcolor='rgba(0,0,0,0.0)'), autosize=False, 
    width=900, height=700, plot_bgcolor="rgba(229,236,246,0.7)",
    margin=dict(l=10, r=10, b=10, t=10), font=dict(size=16, family="Arial")
)

fig.write_image("figures/fig_performance_overall.svg", width=900, height=700)
fig.show()